# 22. PM10 예측: LSTM과 단순 Baseline 비교하기

이번 장에서는 LSTM 모델이 정말 의미가 있는지 확인합니다.

비교할 모델:

```text
1. naive baseline: 직전 값을 그대로 다음 값으로 예측
2. mean baseline: 과거 window 평균을 다음 값으로 예측
3. LSTM: 과거 window 패턴을 학습해 다음 값 예측
```

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import Input, LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping

## 2. 평가 함수 만들기

In [ ]:
def regression_metrics(y_true, y_pred):
    """회귀 예측 결과의 MAE와 RMSE를 계산합니다."""
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return mae, rmse

## 3. PM10 데이터 준비

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\seoul_pm10.csv")
TARGET_AREA = "강남구"

df = pd.read_csv(DATA_PATH, encoding="cp949")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

area_df = df[df["area"] == TARGET_AREA].copy()
area_df = area_df.sort_values("date")
area_df["pm10_filled"] = area_df["pm10"].ffill()
area_df = area_df.dropna(subset=["pm10_filled"])

pm10_values = area_df["pm10_filled"].values.reshape(-1, 1)

print("선택 지역:", TARGET_AREA)
print("데이터 길이:", len(pm10_values))

## 4. 시간 순서 split과 스케일링

In [ ]:
split_index = int(len(pm10_values) * 0.8)

train_values = pm10_values[:split_index]
val_values = pm10_values[split_index:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_values)
val_scaled = scaler.transform(val_values)

print("train 길이:", len(train_scaled))
print("validation 길이:", len(val_scaled))

## 5. window 데이터셋 만들기

In [ ]:
def make_window_dataset(values, window_size):
    """과거 window_size개 값을 X로, 다음 값을 y로 만듭니다."""
    
    X = []
    y = []
    
    for i in range(len(values) - window_size):
        X.append(values[i:i + window_size])
        y.append(values[i + window_size])
    
    return np.array(X), np.array(y)


WINDOW_SIZE = 24

X_train, y_train = make_window_dataset(train_scaled, WINDOW_SIZE)
X_val, y_val = make_window_dataset(val_scaled, WINDOW_SIZE)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

## 6. 실제 정답을 원래 단위로 준비

비교는 원래 PM10 단위에서 하는 것이 해석하기 쉽습니다.

In [ ]:
y_val_original = scaler.inverse_transform(y_val)

print("검증 정답 shape:", y_val_original.shape)
print("정답 앞 5개:", y_val_original[:5].ravel())

## 7. Naive Baseline 만들기

naive baseline은 window의 마지막 값을 다음 값 예측으로 사용합니다.

```text
과거 24시간 중 마지막 시간 PM10 = 다음 시간 예측값
```

In [ ]:
# X_val[:, -1, :]는 각 window의 마지막 시간 값을 꺼냅니다.
naive_pred_scaled = X_val[:, -1, :]

# 원래 PM10 단위로 되돌립니다.
naive_pred_original = scaler.inverse_transform(naive_pred_scaled)

naive_mae, naive_rmse = regression_metrics(y_val_original, naive_pred_original)

print(f"Naive baseline MAE: {naive_mae:.2f}")
print(f"Naive baseline RMSE: {naive_rmse:.2f}")

## 8. Mean Baseline 만들기

mean baseline은 과거 window 평균을 다음 값 예측으로 사용합니다.

In [ ]:
# axis=1은 시간축 방향으로 평균을 낸다는 뜻입니다.
mean_pred_scaled = X_val.mean(axis=1)

mean_pred_original = scaler.inverse_transform(mean_pred_scaled)

mean_mae, mean_rmse = regression_metrics(y_val_original, mean_pred_original)

print(f"Mean baseline MAE: {mean_mae:.2f}")
print(f"Mean baseline RMSE: {mean_rmse:.2f}")

## 9. LSTM 모델 만들기

In [ ]:
model = Sequential([
    Input(shape=(WINDOW_SIZE, 1)),
    LSTM(32),
    Dropout(0.2),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

## 10. LSTM 학습

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

## 11. LSTM 예측

In [ ]:
lstm_pred_scaled = model.predict(X_val, verbose=0)
lstm_pred_original = scaler.inverse_transform(lstm_pred_scaled)

lstm_mae, lstm_rmse = regression_metrics(y_val_original, lstm_pred_original)

print(f"LSTM MAE: {lstm_mae:.2f}")
print(f"LSTM RMSE: {lstm_rmse:.2f}")

## 12. 결과표 만들기

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model": "naive_last_value",
        "MAE": naive_mae,
        "RMSE": naive_rmse,
    },
    {
        "model": "mean_24h",
        "MAE": mean_mae,
        "RMSE": mean_rmse,
    },
    {
        "model": "lstm",
        "MAE": lstm_mae,
        "RMSE": lstm_rmse,
    },
])

comparison_df = comparison_df.sort_values("MAE")
comparison_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(comparison_df["model"], comparison_df["MAE"])
plt.title("MAE Comparison")
plt.ylabel("MAE in original PM10 scale")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 13. 예측 그래프 비교

처음 200개 검증 시점에 대해 실제값, naive, mean, LSTM 예측을 함께 그립니다.

In [ ]:
PLOT_COUNT = 200

plt.figure(figsize=(12, 5))
plt.plot(y_val_original[:PLOT_COUNT], label="actual", linewidth=2)
plt.plot(naive_pred_original[:PLOT_COUNT], label="naive", alpha=0.8)
plt.plot(mean_pred_original[:PLOT_COUNT], label="mean 24h", alpha=0.8)
plt.plot(lstm_pred_original[:PLOT_COUNT], label="lstm", alpha=0.8)
plt.title("Actual vs Baselines vs LSTM")
plt.xlabel("Validation time step")
plt.ylabel("PM10")
plt.legend()
plt.tight_layout()
plt.show()

## 14. LSTM 개선율 계산

In [ ]:
# naive baseline 대비 LSTM이 MAE를 얼마나 줄였는지 계산합니다.
improvement_vs_naive = (naive_mae - lstm_mae) / naive_mae * 100

# mean baseline 대비 LSTM이 MAE를 얼마나 줄였는지 계산합니다.
improvement_vs_mean = (mean_mae - lstm_mae) / mean_mae * 100

print(f"Naive 대비 LSTM MAE 개선율: {improvement_vs_naive:.2f}%")
print(f"Mean 대비 LSTM MAE 개선율: {improvement_vs_mean:.2f}%")

## 15. 해석 문장 만들기

In [ ]:
print("해석 템플릿")
print("- naive baseline은 직전 PM10 값을 다음 값으로 그대로 사용하는 단순 기준선이다.")
print("- mean baseline은 과거 24시간 평균을 다음 값으로 사용하는 기준선이다.")
print("- LSTM 모델은 이 기준선들과 같은 검증 구간에서 MAE/RMSE로 비교했다.")
print("- 딥러닝 모델이 baseline보다 충분히 개선되는지 확인해야 모델 사용 근거가 생긴다.")
print("- 만약 개선 폭이 작다면 추가 특성, 더 긴 데이터 기간, 다른 모델 구조가 필요할 수 있다.")

## 16. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. 딥러닝 모델은 단순 baseline과 비교해야 한다.
2. naive baseline은 직전 값을 다음 값으로 예측한다.
3. mean baseline은 과거 window 평균을 다음 값으로 예측한다.
4. LSTM이 baseline보다 나은지 MAE/RMSE로 비교한다.
5. 복잡한 모델이 항상 단순 모델보다 좋은 것은 아니다.
```

## 과제

1. naive baseline이 강할 수 있는 이유를 설명해보세요.
2. LSTM이 naive보다 낮은 MAE를 냈다면 포트폴리오에 어떻게 쓸지 적어보세요.
3. LSTM이 naive보다 높거나 비슷한 MAE를 냈다면 어떻게 해석할지 적어보세요.
4. PM10 예측에 추가하면 좋을 외부 특성을 세 가지 적어보세요.